# Práctica 10 — Word embeddings

**Curso:** NLP Clásico con Python · **Clase 10** · **Tiempo estimado:** 60-75 minutos
**Notebook de clase:** `notebooks_clase/clase10_word_embeddings.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Los ejercicios 1 y 2 usan cifras que puedes calcular a mano; los 3 a 5 usan el extracto del Quijote de `datos/gutenberg/` (~33 000 palabras: **poco**, y por eso los vectores serán ruidosos) y el reto también el corpus grande de cuatro obras (en Colab, sube la carpeta `datos/`). Es español antiguo, no el de hoy (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. Librerías: `numpy`, `gensim`.

---
## Celda 0 — Setup y datos

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("gensim", "gensim")
print("Librerías listas.")

In [ ]:
import os
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import Word2Vec, FastText

warnings.filterwarnings("ignore")
random.seed(42)                          # semilla fija
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def frases_de_texto(texto):
    """Minúsculas, corte en frases y solo letras; descarta las frases de menos de 3 palabras."""
    salida = []
    for trozo in re.split(r"[.!?;:]+\s|\n+", texto.lower()):
        palabras = re.findall(r"[^\W\d_]+", trozo)
        if len(palabras) >= 3:
            salida.append(palabras)
    return salida

def vecinos(modelo, palabra, n=6):
    """Las n palabras más cercanas por coseno."""
    return [w for w, _ in modelo.wv.most_similar(palabra, topn=n)]

def frecuentes(frases, modelo, n=200, largo=4):
    """Las n palabras más frecuentes (de al menos `largo` letras) que están en el vocabulario del modelo."""
    cuenta = Counter(w for f in frases for w in f)
    return [w for w, _ in cuenta.most_common() if len(w) >= largo and w in modelo.wv.key_to_index][:n]

print("Funciones auxiliares cargadas.")

In [ ]:
# datos-es
# Mini-corpus de 4 frases (español) para el ejercicio 2 y vocabulario de 9 palabras
FRASES_P = [["el", "rey", "ama", "a", "la", "reina"], ["la", "reina", "ama", "al", "rey"],
            ["el", "pastor", "ama", "a", "la", "pastora"], ["la", "pastora", "ama", "al", "pastor"]]
VOCAB_P = ["rey", "reina", "pastor", "pastora", "el", "la", "ama", "a", "al"]
RUTA_PEQUENO = "gutenberg/quijote_cap01-14.txt"
RUTA_GRANDE = "gutenberg/embeddings_es_obras.txt"

In [ ]:
with open(cargar_datos(RUTA_PEQUENO), encoding="utf-8") as f:
    PEQUENO = frases_de_texto(f.read())
print("Extracto pequeño del Quijote: frases", len(PEQUENO), "| palabras", sum(map(len, PEQUENO)))
assert 1000 < len(PEQUENO) < 2000

## Ejercicio 1 — La similitud del coseno

Completa `coseno_p(a, b)`: `a·b / (‖a‖ ‖b‖)`, con NumPy. Antes de programar, a mano: `a = (1, 2, 2)` y `b = (2, 1, 2)` dan `a·b = 2 + 2 + 4 = 8` y `‖a‖ = ‖b‖ = 3`, luego `cos = 8/9 ≈ 0,889`.

In [ ]:
def coseno_p(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return None    # TODO: producto escalar dividido por el producto de las normas (np.dot y np.linalg.norm)

# Comprobación automática
_c = coseno_p([1, 2, 2], [2, 1, 2])
assert _c is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_c, 8 / 9), "a mano: 8/9 ≈ 0,889"
assert np.isclose(coseno_p([3, 4], [30, 40]), 1.0) and np.isclose(coseno_p([3, 4], [-4, 3]), 0.0) and np.isclose(coseno_p([3, 4], [-3, -4]), -1.0)
assert np.isclose(coseno_p([1, 2, 2], [2, 1, 2]), coseno_p([2, 4, 4], [1, 1, 1] + np.array([1, 0, 1])))   # no depende de la longitud
print("Ejercicio 1 correcto ✔", round(_c, 3))

## Ejercicio 2 — Matriz de coocurrencia (vectores dispersos)

Completa `matriz_coocurrencia(frases, vocab, ventana=1)`: matriz cuadrada de recuentos (fila = palabra, columna = palabra de contexto) contando, para cada palabra, las que están a distancia `ventana` o menos **en la misma frase** (sin contarse a sí misma en su propia posición). Solo se cuentan las palabras que están en `vocab`.

A mano, con `FRASES_P` y ventana 1: `rey` aparece en `el rey ama` (vecinas `el` y `ama`) y en `ama al rey` (vecina `al`), luego su fila tiene 1 en `el`, 1 en `ama` y 1 en `al`, y ceros en el resto.

In [ ]:
def matriz_coocurrencia(frases, vocab, ventana=1):
    indice = {p: i for i, p in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)), dtype=int)
    for frase in frases:
        for i, p in enumerate(frase):
            pass    # TODO: para cada posición j a distancia <= ventana de i (j distinto de i, dentro de la frase), suma 1 en M[indice[p], indice[frase[j]]] si ambas están en vocab
    return M

# Comprobación automática
def _cos(a, b):
    return a @ b / np.linalg.norm(a) / np.linalg.norm(b)

_M1 = matriz_coocurrencia(FRASES_P, VOCAB_P, 1)
_ix = {p: i for i, p in enumerate(VOCAB_P)}
assert _M1.sum() > 0, "Completa las líneas marcadas con TODO"
assert _M1.shape == (9, 9) and (_M1 == _M1.T).all(), "la matriz debe ser cuadrada y simétrica"
assert [int(_M1[_ix["rey"], _ix[w]]) for w in VOCAB_P[4:5] + ["ama", "al", "reina"]] == [1, 1, 1, 0]      # el cálculo a mano
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["pastor"]]), 1.0)          # mismos contextos con ventana 1
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["reina"]]), 1 / np.sqrt(15))
_M2 = matriz_coocurrencia(FRASES_P, VOCAB_P, 2)
assert _M2.sum() > _M1.sum() and np.isclose(_cos(_M2[_ix["rey"]], _M2[_ix["reina"]]), 4 / 7)   # ventana mayor: más coocurrencias
print("Ejercicio 2 correcto ✔", int(_M1.sum()), int(_M2.sum()))

## Ejercicio 3 — Entrenar Word2Vec de forma reproducible

Completa `entrenar_p(frases, sg, window, seed)`: un `Word2Vec` con `vector_size=30`, `min_count=3`, `epochs=5`, `workers=1`, la `seed` dada (con un solo hilo, el resultado es el mismo en cada ejecución), y con `sg` y `window` como parámetros (`sg=1` skip-gram, `sg=0` CBOW). Se entrena sobre `PEQUENO` (el extracto del Quijote: ~33 000 palabras, poco).

In [ ]:
def entrenar_p(frases, sg, window, seed):
    return None    # TODO: Word2Vec(frases, vector_size=30, window=window, min_count=3, sg=sg, epochs=5, workers=1, seed=seed)

# Comprobación automática
_a = entrenar_p(PEQUENO, 1, 5, 1)
assert _a is not None, "Completa las líneas marcadas con TODO"
_b = entrenar_p(PEQUENO, 1, 5, 1)
_c = entrenar_p(PEQUENO, 0, 3, 2)
assert _a.wv.vector_size == 30 and _a.sg == 1 and _c.sg == 0 and _c.window == 3, "revisa vector_size, sg y window"
assert _a.wv.key_to_index == _b.wv.key_to_index and np.allclose(_a.wv.vectors, _b.wv.vectors), "mismos datos y misma semilla deben dar el mismo modelo (workers=1)"
assert not np.allclose(_a.wv.vectors, entrenar_p(PEQUENO, 1, 5, 2).wv.vectors), "otra semilla debe dar otros vectores"
print("Ejercicio 3 correcto ✔", "vocabulario:", len(_a.wv))

## Ejercicio 4 — Medir el ruido: solape entre dos modelos

Completa `solape_p(m1, m2, palabras, k=10)`: para cada palabra, la fracción de sus `k` vecinos más próximos (usa la función `vecinos(modelo, palabra, k)` ya definida) que **coinciden** entre los dos modelos; devuelve la media. A mano: si `m1` da `[a, b, c, d]` y `m2` da `[a, b, x, y]` con `k = 4`, coinciden 2 de 4 → 0,5.

In [ ]:
def solape_p(m1, m2, palabras, k=10):
    valores = []
    for w in palabras:
        pass    # TODO: calcula la fracción de coincidencias entre vecinos(m1, w, k) y vecinos(m2, w, k) y añádela a valores
    return float(np.mean(valores)) if valores else None

# Comprobación automática
class _Falso:                                     # modelo de juguete que devuelve vecinos fijos
    def __init__(self, lista):
        self.wv = self
        self._lista = lista
    def most_similar(self, palabra, topn=10):
        return [(w, 1.0) for w in self._lista[:topn]]

_f1, _f2 = _Falso(["a", "b", "c", "d"]), _Falso(["a", "b", "x", "y"])
_r = solape_p(_f1, _f2, ["p"], 4)
assert _r is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_r, 0.5) and np.isclose(solape_p(_f1, _f1, ["p", "q"], 4), 1.0), "a mano: 2 de 4 = 0,5"
_m1 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
_m2 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=2)
_frec = frecuentes(PEQUENO, _m1, 50)
_mismo, _otro = solape_p(_m1, _m1, _frec, 10), solape_p(_m1, _m2, _frec, 10)
assert np.isclose(_mismo, 1.0) and 0 <= _otro < 1
print("Ejercicio 4 correcto ✔", "solape entre semillas en el extracto pequeño:", round(_otro, 3))

## Ejercicio 5 — fastText y palabras fuera de vocabulario

Se entrena un `FastText` sobre `PEQUENO` (celda de abajo). Completa `vector_de(modelo, palabra)`: devuelve la pareja `(esta_en_vocabulario, vector)`, donde `esta_en_vocabulario` dice si `palabra` está en `modelo.wv.key_to_index` y `vector` es `modelo.wv[palabra]`. Después comprueba que `Word2Vec` lanza un `KeyError` con la errata `caballeroo` y que fastText **sí** le da un vector.

In [ ]:
FT_P = FastText(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1, min_n=3, max_n=5)

def vector_de(modelo, palabra):
    esta = None      # TODO: ¿está palabra en modelo.wv.key_to_index?
    vector = None    # TODO: modelo.wv[palabra]
    return esta, vector

# Comprobación automática
_esta, _vec = vector_de(FT_P, "caballeroo")
assert _esta is not None and _vec is not None, "Completa las líneas marcadas con TODO"
assert _esta is False and _vec.shape == (30,), "la errata no está en el vocabulario, pero fastText le da un vector de 30 dimensiones"
assert vector_de(FT_P, "caballero")[0] is True
_w2v = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
try:
    _w2v.wv["caballeroo"]
    _lanza = False
except KeyError:
    _lanza = True
assert _lanza, "Word2Vec debería lanzar KeyError"
_cos_e = float(np.dot(_vec, FT_P.wv["caballero"]) / (np.linalg.norm(_vec) * np.linalg.norm(FT_P.wv["caballero"])))
assert -1 <= _cos_e <= 1
print("Ejercicio 5 correcto ✔", "cos(caballeroo, caballero) =", round(_cos_e, 3))

---
## Reto final — ¿Más texto, menos ruido?

Con **tus** funciones `entrenar_p` (ejercicio 3) y `solape_p` (ejercicio 4) y con `frecuentes(frases, modelo, n)`:

1. Completa `solape_semillas(frases, n=100)`: entrena dos modelos skip-gram (`window=5`) con las semillas 1 y 2, toma las `n` palabras más frecuentes (`frecuentes`, calculadas sobre el modelo de la semilla 1) y devuelve el `solape_p` a 10 vecinos.
2. Calcúlalo para tres tamaños: el extracto pequeño (`PEQUENO`), el **10 %** y el **50 %** de las frases del corpus español grande (`GRANDE`, cuatro obras, ~758 000 palabras). Guarda los resultados en `res_tamano` (claves `"pequeno"`, `"10%"`, `"50%"`).
3. Escribe abajo tus conclusiones: ¿sube el solape con el texto? ¿Es suficiente para fiarse de un vecino aislado?

Tarda unos 15-20 segundos. *(Ejemplo ilustrativo: con otros corpus y parámetros las cifras cambian.)*

In [ ]:
with open(cargar_datos(RUTA_GRANDE), encoding="utf-8") as f:
    GRANDE = frases_de_texto(f.read())

def solape_semillas(frases, n=100):
    return None    # TODO: entrena con entrenar_p (sg=1, window=5, semillas 1 y 2) y devuelve solape_p sobre frecuentes(frases, m1, n) a 10 vecinos

res_tamano = {}
# TODO: rellena res_tamano["pequeno"], res_tamano["10%"] y res_tamano["50%"] con solape_semillas (las fracciones son las primeras frases de GRANDE)

# Comprobación automática
assert len(res_tamano) == 3 and all(v is not None for v in res_tamano.values()), "Completa las líneas marcadas con TODO"
assert all(0 <= v <= 1 for v in res_tamano.values()) and set(res_tamano) == {"pequeno", "10%", "50%"}
assert res_tamano["pequeno"] < res_tamano["10%"] and res_tamano["pequeno"] < res_tamano["50%"], "con muy poco texto el solape debería ser el más bajo"
print("Reto final correcto ✔")
print({k: round(v, 3) for k, v in res_tamano.items()})

**Tus conclusiones (escribe 3-4 frases):** ¿por qué el coseno ignora la longitud del vector? ¿Qué cambia entre un vector disperso de coocurrencias y un embedding denso? ¿Por qué `workers=1` y una semilla fija hacen falta para reproducir un modelo? ¿Cómo cambia el solape con el tamaño del corpus y qué implica para interpretar un vecino aislado? ¿Qué gana y qué no gana fastText con una errata?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
def coseno_p(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Comprobación automática
_c = coseno_p([1, 2, 2], [2, 1, 2])
assert _c is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_c, 8 / 9), "a mano: 8/9 ≈ 0,889"
assert np.isclose(coseno_p([3, 4], [30, 40]), 1.0) and np.isclose(coseno_p([3, 4], [-4, 3]), 0.0) and np.isclose(coseno_p([3, 4], [-3, -4]), -1.0)
assert np.isclose(coseno_p([1, 2, 2], [2, 1, 2]), coseno_p([2, 4, 4], [1, 1, 1] + np.array([1, 0, 1])))   # no depende de la longitud
print("Ejercicio 1 correcto ✔", round(_c, 3))

### Solución del ejercicio 2

In [ ]:
def matriz_coocurrencia(frases, vocab, ventana=1):
    indice = {p: i for i, p in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)), dtype=int)
    for frase in frases:
        for i, p in enumerate(frase):
            for j in range(max(0, i - ventana), min(len(frase), i + ventana + 1)):
                if j != i and p in indice and frase[j] in indice:
                    M[indice[p], indice[frase[j]]] += 1
    return M

# Comprobación automática
def _cos(a, b):
    return a @ b / np.linalg.norm(a) / np.linalg.norm(b)

_M1 = matriz_coocurrencia(FRASES_P, VOCAB_P, 1)
_ix = {p: i for i, p in enumerate(VOCAB_P)}
assert _M1.sum() > 0, "Completa las líneas marcadas con TODO"
assert _M1.shape == (9, 9) and (_M1 == _M1.T).all(), "la matriz debe ser cuadrada y simétrica"
assert [int(_M1[_ix["rey"], _ix[w]]) for w in VOCAB_P[4:5] + ["ama", "al", "reina"]] == [1, 1, 1, 0]      # el cálculo a mano
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["pastor"]]), 1.0)          # mismos contextos con ventana 1
assert np.isclose(_cos(_M1[_ix["rey"]], _M1[_ix["reina"]]), 1 / np.sqrt(15))
_M2 = matriz_coocurrencia(FRASES_P, VOCAB_P, 2)
assert _M2.sum() > _M1.sum() and np.isclose(_cos(_M2[_ix["rey"]], _M2[_ix["reina"]]), 4 / 7)   # ventana mayor: más coocurrencias
print("Ejercicio 2 correcto ✔", int(_M1.sum()), int(_M2.sum()))

### Solución del ejercicio 3

In [ ]:
def entrenar_p(frases, sg, window, seed):
    return Word2Vec(frases, vector_size=30, window=window, min_count=3, sg=sg, epochs=5, workers=1, seed=seed)

# Comprobación automática
_a = entrenar_p(PEQUENO, 1, 5, 1)
assert _a is not None, "Completa las líneas marcadas con TODO"
_b = entrenar_p(PEQUENO, 1, 5, 1)
_c = entrenar_p(PEQUENO, 0, 3, 2)
assert _a.wv.vector_size == 30 and _a.sg == 1 and _c.sg == 0 and _c.window == 3, "revisa vector_size, sg y window"
assert _a.wv.key_to_index == _b.wv.key_to_index and np.allclose(_a.wv.vectors, _b.wv.vectors), "mismos datos y misma semilla deben dar el mismo modelo (workers=1)"
assert not np.allclose(_a.wv.vectors, entrenar_p(PEQUENO, 1, 5, 2).wv.vectors), "otra semilla debe dar otros vectores"
print("Ejercicio 3 correcto ✔", "vocabulario:", len(_a.wv))

### Solución del ejercicio 4

In [ ]:
def solape_p(m1, m2, palabras, k=10):
    valores = []
    for w in palabras:
        valores.append(len(set(vecinos(m1, w, k)) & set(vecinos(m2, w, k))) / k)
    return float(np.mean(valores)) if valores else None

# Comprobación automática
class _Falso:                                     # modelo de juguete que devuelve vecinos fijos
    def __init__(self, lista):
        self.wv = self
        self._lista = lista
    def most_similar(self, palabra, topn=10):
        return [(w, 1.0) for w in self._lista[:topn]]

_f1, _f2 = _Falso(["a", "b", "c", "d"]), _Falso(["a", "b", "x", "y"])
_r = solape_p(_f1, _f2, ["p"], 4)
assert _r is not None, "Completa las líneas marcadas con TODO"
assert np.isclose(_r, 0.5) and np.isclose(solape_p(_f1, _f1, ["p", "q"], 4), 1.0), "a mano: 2 de 4 = 0,5"
_m1 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
_m2 = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=2)
_frec = frecuentes(PEQUENO, _m1, 50)
_mismo, _otro = solape_p(_m1, _m1, _frec, 10), solape_p(_m1, _m2, _frec, 10)
assert np.isclose(_mismo, 1.0) and 0 <= _otro < 1
print("Ejercicio 4 correcto ✔", "solape entre semillas en el extracto pequeño:", round(_otro, 3))

### Solución del ejercicio 5

In [ ]:
FT_P = FastText(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1, min_n=3, max_n=5)

def vector_de(modelo, palabra):
    esta = palabra in modelo.wv.key_to_index
    vector = modelo.wv[palabra]
    return esta, vector

# Comprobación automática
_esta, _vec = vector_de(FT_P, "caballeroo")
assert _esta is not None and _vec is not None, "Completa las líneas marcadas con TODO"
assert _esta is False and _vec.shape == (30,), "la errata no está en el vocabulario, pero fastText le da un vector de 30 dimensiones"
assert vector_de(FT_P, "caballero")[0] is True
_w2v = Word2Vec(PEQUENO, vector_size=30, window=5, min_count=3, sg=1, epochs=5, workers=1, seed=1)
try:
    _w2v.wv["caballeroo"]
    _lanza = False
except KeyError:
    _lanza = True
assert _lanza, "Word2Vec debería lanzar KeyError"
_cos_e = float(np.dot(_vec, FT_P.wv["caballero"]) / (np.linalg.norm(_vec) * np.linalg.norm(FT_P.wv["caballero"])))
assert -1 <= _cos_e <= 1
print("Ejercicio 5 correcto ✔", "cos(caballeroo, caballero) =", round(_cos_e, 3))

### Solución del reto final

In [ ]:
with open(cargar_datos(RUTA_GRANDE), encoding="utf-8") as f:
    GRANDE = frases_de_texto(f.read())

def solape_semillas(frases, n=100):
    m1, m2 = entrenar_p(frases, 1, 5, 1), entrenar_p(frases, 1, 5, 2)
    return solape_p(m1, m2, frecuentes(frases, m1, n), 10)

res_tamano = {"pequeno": solape_semillas(PEQUENO),
              "10%": solape_semillas(GRANDE[: len(GRANDE) // 10]),
              "50%": solape_semillas(GRANDE[: len(GRANDE) // 2])}

# Comprobación automática
assert len(res_tamano) == 3 and all(v is not None for v in res_tamano.values()), "Completa las líneas marcadas con TODO"
assert all(0 <= v <= 1 for v in res_tamano.values()) and set(res_tamano) == {"pequeno", "10%", "50%"}
assert res_tamano["pequeno"] < res_tamano["10%"] and res_tamano["pequeno"] < res_tamano["50%"], "con muy poco texto el solape debería ser el más bajo"
print("Reto final correcto ✔")
print({k: round(v, 3) for k, v in res_tamano.items()})

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto no te ha quedado claro (coseno, coocurrencia y vectores dispersos, skip-gram frente a CBOW, ruido entre semillas, fastText y n-gramas)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*